# GPU-Accelerated Plant Disease Detection\nColab GPU execution notebook.\n

## 1. Colab GPU Verification\n

In [ ]:
!nvidia-smi\n

In [ ]:
import sys
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    cuda_available = torch.cuda.is_available()
    print(f"CUDA available: {cuda_available}")
    if cuda_available:
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU count: {torch.cuda.device_count()}")
    else:
        print("WARNING: CUDA GPU is not available.")
        print("Please change your Colab runtime to GPU (Runtime -> Change runtime type).")
        sys.exit("Stopping notebook execution. Enable a GPU runtime first.")
except ImportError:
    print("PyTorch is not installed. We will install it in the next cell.")
\n

## 2. Repository Setup\n

In [ ]:
import os
repo_url = "https://github.com/Prathamesh-98/gpu-plant-disease-detection.git"
repo_dir = "gpu-plant-disease-detection"

if not os.path.exists(repo_dir):
    print("Cloning repository...")
    !git clone {repo_url}
else:
    print("Repository exists. Pulling latest...")
    !cd {repo_dir} && git pull

import sys
if f"/content/{repo_dir}" not in sys.path:
    sys.path.append(f"/content/{repo_dir}")
\n

In [ ]:
!pip install -r {repo_dir}/requirements.txt\n

## 3. Dataset Setup\nPlace your dataset inside `gpu-plant-disease-detection/data/raw/`\n

In [ ]:
# Optional Dataset extraction
import os
# dataset_zip = "/content/drive/MyDrive/dataset.zip"
# target_dir = f"/content/{repo_dir}/data/raw/"
# os.makedirs(target_dir, exist_ok=True)
# !unzip -q {dataset_zip} -d {target_dir}
print("Ensure dataset is mounted or extracted to data/raw/")
\n

## 4. Dataset Inspection\n

In [ ]:
import os
import csv
import matplotlib.pyplot as plt
from pathlib import Path

raw_dir = Path(f"/content/{repo_dir}/data/raw")
results_dir = Path(f"/content/{repo_dir}/results")
for subdir in ['figures', 'benchmarks', 'metrics', 'predictions']:
    (results_dir / subdir).mkdir(parents=True, exist_ok=True)

classes = [d for d in raw_dir.iterdir() if d.is_dir()]
total_images = 0
class_counts = {}

for c_dir in classes:
    count = len([f for f in c_dir.iterdir() if f.is_file()])
    class_counts[c_dir.name] = count
    total_images += count

print(f"Dataset path: {raw_dir}")
print(f"Number of classes: {len(classes)}")
print(f"Total images: {total_images}")

if total_images > 0:
    with open(results_dir / "dataset_report.csv", "w", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['class_name', 'image_count'])
        for c, count in class_counts.items():
            writer.writerow([c, count])
            
    plt.figure(figsize=(10, 6))
    plt.bar(class_counts.keys(), class_counts.values())
    plt.xticks(rotation=90)
    plt.ylabel('Image Count')
    plt.title('Dataset Class Distribution')
    plt.tight_layout()
    plt.savefig(results_dir / "figures/class_distribution.png")
    plt.show()
\n

## 5. Training Configuration\n

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.20
SEED = 42
\n

## 6. Model Initialization\n

In [ ]:
import torch
from src.dataset import get_dataloaders
from src.model import create_model
from src.config import NORM_MEAN, NORM_STD

if total_images > 0:
    train_loader, val_loader, class_names = get_dataloaders(
        data_dir=raw_dir, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, 
        validation_split=VALIDATION_SPLIT, seed=SEED
    )
    num_classes = len(class_names)
    print(f"Initializing EfficientNet-B0 for {num_classes} classes...")
    model = create_model(num_classes=num_classes, pretrained=True)
\n

## 7. Real GPU Training\n

In [ ]:
import pandas as pd
import torch.optim as optim
import torch.nn as nn
from src.train import train_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

if device.type == 'cuda' and total_images > 0:
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    model, history = train_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        criterion=criterion, optimizer=optimizer, num_epochs=EPOCHS, device=device
    )
    pd.DataFrame(history).to_csv(results_dir / "training_history.csv", index=False)
else:
    print("Training skipped. Either no GPU or no dataset found.")
\n

## 8. Training Curves\n

In [ ]:
if 'history' in locals() and total_images > 0:
    epochs_range = range(1, EPOCHS + 1)
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history['train_loss'], label='Train Loss')
    plt.plot(epochs_range, history['val_loss'], label='Val Loss')
    plt.legend()
    plt.title('Loss Curve')
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history['train_acc'], label='Train Accuracy')
    plt.plot(epochs_range, history['val_acc'], label='Val Accuracy')
    plt.legend()
    plt.title('Accuracy Curve')
    
    plt.savefig(results_dir / "figures/training_curves.png")
    plt.show()
\n

## 9. Model Checkpoint\n

In [ ]:
models_dir = Path(f"/content/{repo_dir}/models")
models_dir.mkdir(exist_ok=True)

if total_images > 0:
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'class_names': class_names,
        'num_classes': num_classes,
        'image_size': IMAGE_SIZE,
        'model_name': 'efficientnet_b0',
        'config': {'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE}
    }
    ckpt_path = models_dir / "plant_disease_efficientnet_b0.pth"
    torch.save(checkpoint, ckpt_path)
    print(f"Checkpoint saved to {ckpt_path}")
\n

## 10. Sample Predictions & Confusion Matrix\n

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

if total_images > 0 and device.type == 'cuda':
    all_preds, all_trues, all_confs = [], [], []
    model.eval()
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            confs, preds = torch.max(probs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_trues.extend(labels.cpu().numpy())
            all_confs.extend(confs.cpu().numpy())

    # Classification Report
    report = classification_report(all_trues, all_preds, target_names=class_names, output_dict=True)
    pd.DataFrame(report).transpose().to_csv(results_dir / "metrics" / "classification_report.csv")
    
    # Confusion Matrix
    cm = confusion_matrix(all_trues, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('Confusion Matrix')
    plt.savefig(results_dir / "figures/confusion_matrix.png")
    plt.show()
    
    # Sample Predictions CSV
    df_preds = pd.DataFrame({
        'true_class': [class_names[i] for i in all_trues],
        'predicted_class': [class_names[i] for i in all_preds],
        'confidence': all_confs,
        'correct': np.array(all_trues) == np.array(all_preds)
    })
    df_preds.to_csv(results_dir / "predictions/sample_predictions.csv", index=False)
    
    # Sample Visuals
    def imshow(inp, title=None):
        inp = inp.numpy().transpose((1, 2, 0))
        mean = np.array(NORM_MEAN)
        std = np.array(NORM_STD)
        inp = std * inp + mean
        inp = np.clip(inp, 0, 1)
        plt.imshow(inp)
        if title is not None:
            plt.title(title, color='green' if 'True' in title else 'red')

    inputs, labels = next(iter(val_loader))
    plt.figure(figsize=(15, 5))
    for i in range(min(5, len(inputs))):
        plt.subplot(1, 5, i+1)
        pred_label = class_names[all_preds[i]]
        true_label = class_names[labels[i].item()]
        imshow(inputs[i].cpu(), title=f"P:{pred_label}\nT:{true_label}" if pred_label==true_label else f"P:{pred_label}\nT:{true_label} [False]")
        plt.axis('off')
    plt.savefig(results_dir / "figures/sample_predictions.png")
    plt.show()
\n

## 11. CPU vs GPU Benchmark\n

In [ ]:
from src.benchmark import run_benchmark

batch_sizes = [8, 16, 32, 64]
benchmark_results = []

def safe_benchmark(model, dataloader, dev_str, bs):
    try:
        res = run_benchmark(model, dataloader, torch.device(dev_str))
        res['batch_size'] = bs
        return res
    except RuntimeError as e:
        if 'out of memory' in str(e):
            print(f"OOM on {dev_str} with batch size {bs}")
            torch.cuda.empty_cache()
            return None
        raise e

if total_images > 0 and torch.cuda.is_available():
    for bs in batch_sizes:
        _, bs_val_loader, _ = get_dataloaders(
            data_dir=raw_dir, image_size=IMAGE_SIZE, batch_size=bs, validation_split=VALIDATION_SPLIT, seed=SEED
        )
        
        gpu_res = safe_benchmark(model, bs_val_loader, 'cuda', bs)
        if gpu_res: benchmark_results.append(gpu_res)
            
        cpu_res = safe_benchmark(model, bs_val_loader, 'cpu', bs)
        if cpu_res: benchmark_results.append(cpu_res)

if benchmark_results:
    df_bench = pd.DataFrame(benchmark_results)
    
    cpu_df = df_bench[df_bench['device'] == 'cpu'].set_index('batch_size')
    gpu_df = df_bench[df_bench['device'] == 'cuda'].set_index('batch_size')
    
    speedups = []
    for bs in gpu_df.index:
        if bs in cpu_df.index:
            speedup = cpu_df.loc[bs, 'total_time_seconds'] / gpu_df.loc[bs, 'total_time_seconds']
            speedups.append({'batch_size': bs, 'speedup': speedup})
            
    df_speedup = pd.DataFrame(speedups)
    
    df_bench.to_csv(results_dir / "benchmarks/cpu_vs_gpu_results.csv", index=False)
    df_speedup.to_csv(results_dir / "benchmarks/gpu_speedups.csv", index=False)
    
    plt.figure(figsize=(10, 6))
    if not cpu_df.empty: plt.plot(cpu_df.index, cpu_df['images_per_second'], marker='o', label='CPU')
    if not gpu_df.empty: plt.plot(gpu_df.index, gpu_df['images_per_second'], marker='s', label='GPU (CUDA)')
    plt.title('Throughput vs Batch Size')
    plt.xlabel('Batch Size')
    plt.ylabel('Images / Second')
    plt.legend()
    plt.grid(True)
    plt.savefig(results_dir / "figures/cpu_gpu_throughput.png")
    plt.show()
    print("Benchmark complete and saved.")
\n

## 12. GPU Environment Evidence\n

In [ ]:
if torch.cuda.is_available():
    env_info = [
        f"GPU Name: {torch.cuda.get_device_name(0)}",
        f"GPU Memory Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB",
        f"CUDA Version: {torch.version.cuda}",
        f"PyTorch Version: {torch.__version__}"
    ]
    with open(results_dir / "gpu_environment.txt", "w") as f:
        f.write("\n".join(env_info))
    print("GPU environment evidence saved.")
\n